# SmolVLM-256M: Attention Implementation

This notebook compares the attention implementations of SmolVLM-256M: the default, eager and sdpa. It also tests sdpa with the mask that eager uses.

In [ ]:
import copy
import gc
import json
import re
import subprocess
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, StaticCache
from transformers.modeling_outputs import BaseModelOutput

ROOT = next(p for p in (Path(".."), Path("."), Path("/content/edge-ai-smol-lvm-250-optimization"))
            if (p / "frame.jpg").exists())
IMAGE = ROOT / "frame.jpg"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
PROMPT = "Describe any visible defects."
PREPROCESS = {"default": (None, "cpu"), "slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}


def timed(fn):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) * 1e3


def mean_ms(fn, runs=5):
    fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(runs):
        fn()
        torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1e3 / runs


def until_eos(ids, eos):
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


def set_attn(model, impl):
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation(impl)
    else:
        for c in (model.config.text_config, model.config.vision_config):
            c._attn_implementation = impl


def load(attn=None, fast=None):
    kw = {} if attn is None else {"attn_implementation": attn}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **({} if fast is None else {"use_fast": fast}))
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, **kw).to("cuda").eval()
    return model, proc


def eos_ids(tok):
    return {tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_utterance>")}


def chat_text(proc, prompt):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    return proc.apply_chat_template(msgs, add_generation_prompt=True)


def make_inputs(proc, image, prompt, image_device="cpu", size=None, extra=None):
    img = Image.open(image).convert("RGB") if isinstance(image, (str, Path)) else image
    dev = {} if image_device == "cpu" else {"device": image_device}
    ik = {**({} if size is None else {"size": {"longest_edge": size}}), **dev, **(extra or {})}
    text = chat_text(proc, prompt)
    try:
        out = proc(text=text, images=[img], return_tensors="pt", images_kwargs=ik)
    except (TypeError, ValueError):
        out = proc(text=text, images=[img], return_tensors="pt", **ik)
    out = out.to("cuda")
    out["pixel_values"] = out["pixel_values"].to(torch.float16)
    return out


def bare_generate(model, inputs, k, stop=True):
    extra = {} if stop else {"min_new_tokens": k}
    with torch.inference_mode():
        return model.generate(**inputs, max_new_tokens=k, do_sample=False, **extra)


class GraphDecoder:
    def __init__(self, model, max_new=64, max_len=4096, own_mask=False):
        self.model, self.max_new, self.max_len = model, max_new, max_len
        self.cache = StaticCache(config=model.config.text_config, max_cache_len=max_len)
        dev = model.device
        self.ids = torch.zeros(1, 1, dtype=torch.long, device=dev)
        self.pos = torch.zeros(1, dtype=torch.long, device=dev)
        self.idx = torch.zeros(1, dtype=torch.long, device=dev)
        self.out = torch.zeros(max_new, dtype=torch.long, device=dev)
        self.graph = None
        self.steps = max_new
        self.own_mask = own_mask or model.config.text_config._attn_implementation == "eager"

    def mask(self):
        allowed = torch.arange(self.max_len, device=self.pos.device) <= self.pos[:, None]
        dt = self.model.dtype
        return ((~allowed).to(dt) * torch.finfo(dt).min)[:, None, None, :]

    def step(self):
        extra = {"attention_mask": self.mask()} if self.own_mask else {}
        h = self.model.model.text_model(
            input_ids=self.ids, cache_position=self.pos, past_key_values=self.cache, use_cache=True, **extra
        ).last_hidden_state
        nxt = self.model.lm_head(h[:, -1]).argmax(-1, keepdim=True)
        self.ids.copy_(nxt)
        self.out.index_copy_(0, self.idx, nxt.view(1))
        self.pos.add_(1)
        self.idx.add_(1)

    @torch.inference_mode()
    def capture(self):
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                self.pos.fill_(self.max_len - 1)
                self.idx.zero_()
                self.step()
        torch.cuda.current_stream().wait_stream(s)
        self.pos.fill_(self.max_len - 1)
        self.idx.zero_()
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.step()

    @torch.inference_mode()
    def prefill(self, inputs):
        self.cache.reset()
        n = inputs["input_ids"].shape[1]
        logits = self.model(
            **inputs, past_key_values=self.cache, use_cache=True,
            cache_position=torch.arange(n, device=self.model.device),
        ).logits
        first = logits[:, -1].argmax(-1, keepdim=True)
        self.ids.copy_(first)
        self.pos.fill_(n)
        self.idx.zero_()
        return first

    @torch.inference_mode()
    def decode(self, use_graph=True, eos=None):
        self.steps = self.max_new
        for i in range(self.max_new):
            if use_graph:
                self.graph.replay()
            else:
                self.step()
            if eos and self.ids.item() in eos:
                self.steps = i + 1
                break
        return self.out.clone()

    def generate(self, inputs, eos, use_graph=True):
        first = self.prefill(inputs)
        ids = torch.cat([first.view(1), self.decode(use_graph, eos)]).tolist()
        return until_eos(ids, eos)


def time_decode(dec, inputs, eos, runs=5, use_graph=True, stop=False):
    pre, dec_ms, ids = [], [], None
    for i in range(runs + 1):
        first, p = timed(lambda: dec.prefill(inputs))
        out, d = timed(lambda: dec.decode(use_graph, eos if stop else None))
        if i:
            pre.append(p)
            dec_ms.append(d)
        ids = until_eos(torch.cat([first.view(1), out]).tolist(), eos)
    return {"prefill_ms": float(np.mean(pre)), "decode_ms": float(np.mean(dec_ms)), "steps": dec.steps, "ids": ids}


def fit_len(n, max_new):
    return -(-(n + max_new + 1) // 64) * 64

In [ ]:
def attn_run(name, attn=None, own_mask=False, runs=5):
    model, proc = load(attn=attn)
    used = f"{model.config.text_config._attn_implementation}/{model.config.vision_config._attn_implementation}"
    inputs = make_inputs(proc, IMAGE, PROMPT)
    eos = eos_ids(proc.tokenizer)
    dec = GraphDecoder(model, 64, 1024, own_mask)
    dec.capture()
    torch.cuda.reset_peak_memory_stats()
    r = time_decode(dec, inputs, eos, runs)
    peak = torch.cuda.max_memory_allocated() / 2**20
    del model, dec
    gc.collect()
    torch.cuda.empty_cache()
    return {"option": name, "text/vision": used, "prefill ms": round(r["prefill_ms"], 1),
            "decode ms/token": round(r["decode_ms"] / 64, 2), "peak VRAM MB": round(peak), "ids": r["ids"]}


runs = [attn_run("default"), attn_run("eager", "eager"), attn_run("sdpa", "sdpa"),
        attn_run("sdpa+mask", "sdpa", True), attn_run("eager+mask", "eager", True)]
ref = runs[0]["ids"]
table = pd.DataFrame(runs)
table["same answer"] = [r["ids"] == ref for r in runs]
table.drop(columns="ids")

## Log of previous runs

The cells below are the earlier notebook with their outputs. They call the scripts that were replaced by the code above, so they are not run again.

In [11]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/attn_check.py --image {REPO}/frame.jpg --max-len 1024

option   text/vision   prefill ms decode ms/tok peak VRAM MB  same answer
Loading weights: 100% 471/471 [00:00<00:00, 1210.78it/s]
default  sdpa/sdpa          272.8          8.13          873  True
Loading weights: 100% 471/471 [00:00<00:00, 1104.81it/s]
eager    eager/eager        392.8          6.45         2012  True
Loading weights: 100% 471/471 [00:00<00:00, 1427.10it/s]
sdpa     sdpa/sdpa          280.2          8.15          889  True


## Result

The test ran on a Colab GPU. The prompt has 875 tokens, and max_len is 1024. All options give the same answer.

| Option | Text / vision | Prefill (ms) | Decode (ms/token) | Peak VRAM (MB) |
|---|---|---|---|---|
| default | sdpa / sdpa | 272.8 | 8.13 | 873 |
| eager | eager / eager | 392.8 | 6.45 | 2012 |
| sdpa | sdpa / sdpa | 280.2 | 8.15 | 889 |

- The default option already uses `sdpa` for the text model and the vision model. To set `sdpa` changes nothing.
- `sdpa` is 1.4 times faster in prefill (280 ms and 393 ms). It also uses less peak VRAM (889 MB and 2012 MB). The `eager` option probably builds the full attention matrices, and this uses the extra memory.
- `eager` is 1.26 times faster in decode (6.45 ms and 8.15 ms per token). We do not know the cause. The `eager` run uses its own simple mask. The difference can come from the mask and not from the attention.
- For 64 new tokens, the two options take about the same time for prefill and decode (about 802 ms for `sdpa` and 806 ms for `eager`). `eager` is faster only for answers longer than about 66 tokens.
- Keep `sdpa`. The answers are short, and `sdpa` uses less VRAM.
- Next test: run `sdpa` with the same simple mask, to find the cause of the faster decode. A mix could also work: `sdpa` for prefill and `eager` for decode.

In [13]:
# sdpa+mask: sdpa attention with the same simple mask that eager uses
!python {REPO}/attn_check.py --image {REPO}/frame.jpg --max-len 1024 --impls sdpa,sdpa+mask,eager

option     text/vision   prefill ms decode ms/tok peak VRAM MB  same answer
Loading weights: 100% 471/471 [00:00<00:00, 1058.74it/s]
sdpa       sdpa/sdpa          270.4          8.14          873  True
Loading weights: 100% 471/471 [00:00<00:00, 1265.05it/s]
sdpa+mask  sdpa/sdpa          274.3          7.95          881  True
Loading weights: 100% 471/471 [00:00<00:00, 1365.90it/s]
eager      eager/eager        392.7          6.45         2020  True


## Result: mask test

The test ran on the same Colab GPU, with the same image and prompt. All options give the same answer.

| Option | Text / vision | Prefill (ms) | Decode (ms/token) | Peak VRAM (MB) |
|---|---|---|---|---|
| sdpa | sdpa / sdpa | 273.3 | 8.13 | 873 |
| sdpa+mask | sdpa / sdpa | 269.4 | 7.95 | 881 |
| eager | eager / eager | 396.1 | 6.45 | 2020 |

- The simple mask gives `sdpa` only a small gain in decode: 8.13 ms and 7.95 ms per token. The gain is 0.18 ms. The spread between runs is about 0.03 ms.
- The mask is not the main cause. `eager` is still 1.5 ms per token faster than `sdpa+mask` (6.45 ms and 7.95 ms). The attention itself is the cause. We do not know why.
- Prefill and VRAM do not change with the mask. `sdpa` uses about 270 ms and 880 MB. `eager` uses about 396 ms and 2020 MB.
- Next test: `sdpa` for prefill and `eager` for decode. This can save about 1.7 ms for each token. For 64 tokens, this is about 107 ms.